# Who's Worth Calling? — Assignment

**Week 5 · Assignment 5** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026
**Points:** 100 &nbsp;·&nbsp; **Due:** *End of Week 5*

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week5/assignments/week05_retention_calls.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Picks up where Lesson 8 ("Random Forest & XGBoost on Tabular Data") left
off — same dataset, same split, a different decision.*

## Setup

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib xgboost git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

from gbus564_coursework.utils.course_data import load_churn
from gbus564_coursework.utils.assignment_checks import check_close, check_equal, check_in_range, check_shape, check_type

## Before You Start

- **Format — same shape every week:** **A. Extend** (40 pts, scaffolded
  fill-ins on the week's mechanic) → **B. Decide** (30 pts, one modeling
  decision with a number and a justification) → **C. Recommend** (30 pts, a
  stakeholder-facing writeup). Edit only the `# YOUR CODE HERE` blocks and
  the *\<placeholder\>* prose — everything else is given.
- **Self-checks are feedback, not the grade.** The `assert` cells after each
  Part A fill-in are checkpoints: green output means you're on track. They
  don't award points on their own.
- **Submission:** *Restart & Run All*, then download the executed `.ipynb`
  and submit that file — outputs must be present in what you turn in.
- **AI use is expected** (see syllabus). Parts B and C are graded on
  judgment against *your own* results, so generic AI prose won't hold up —
  ground your answer in the numbers your own run produced.

## Summary of Answers

*Fill this in last, after finishing Parts B and C below. It's what gets read
first when this is graded — the point is to make grading possible from this
screen alone, scrolling down only if something looks off.*

| Part | Your Answer |
| --- | --- |
| **B — Decide** (call threshold on XGBoost's score) | _Your answer here._ |
| **C — Recommend** | _Your answer here._ |

## 1. Business Problem

Lesson 8's walkthrough asked *which customers are likely to churn* — and
found XGBoost ties logistic regression for the best test AUC (0.83). It
never asked what to *do* with that ranking. This assignment does: a
retention call isn't free, and a save isn't free money either, so the
right action depends on both the model's score **and** the dollars behind
it.

**Business question:** at what predicted-churn-probability threshold should
Customer Success start making retention calls, given what a call costs and
what a save is worth?

**Why it matters:** call every customer the model flags as risky and most
of the calls are wasted on people who'd have stayed anyway; call no one and
preventable churn — 1,869 customers left in this dataset alone — walks out
the door with no attempt to stop it.

## 2. Data — Part A: Extend

*Same dataset, same split as Lessons 3 and 8 — this has to match exactly
for the model below to be comparable to what Lesson 8 already found. Fill
in the marked block, then run the checkpoint.*

In [ ]:
df = load_churn()
df.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# TODO: same split as Lesson 3/8 — test_size=0.2, random_state=RANDOM_SEED,
# stratify=y — plus a ColumnTransformer that one-hot encodes the categorical
# columns and passes the numeric ones through.
# YOUR CODE HERE


In [ ]:
check_shape(X_train, (X_train.shape[0], X.shape[1]), "X_train has the right number of columns")
check_close(len(X_train) / len(X), 0.8, "train_test_split used test_size=0.2", atol=0.01)

## 3. Model — Part A: Extend

*Baseline is given. Fit the same default XGBoost configuration Lesson 8 started from, so
your numbers are directly comparable to its 0.83 AUC — then refit Lesson 8's
logistic regression alongside it. Part B asks whether the fancier model
actually changes who gets called, so you need both.*

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

In [ ]:
# XGBoost lives in its own package — make sure it is installed before importing it.
%pip install -q xgboost

# TODO: XGBoost, same default hyperparameters as Lesson 8 — n_estimators=300,
# max_depth=3, learning_rate=0.1, random_state=RANDOM_SEED — inside a
# Pipeline with make_preprocessor().
# YOUR CODE HERE


In [ ]:
check_type(model, Pipeline, "model is a fitted Pipeline")
check_equal(hasattr(model, "predict_proba"), True, "model exposes predict_proba (needed for Part B)")

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: Lesson 8's logistic regression — LogisticRegression(C=1.78,
# max_iter=5000, random_state=RANDOM_SEED) — inside a Pipeline with
# make_preprocessor(), fit on the training rows. Call it logreg.
# YOUR CODE HERE


In [ ]:
check_type(logreg, Pipeline, "logreg is a fitted Pipeline")
check_type(logreg.named_steps["clf"], LogisticRegression, "logreg's final step is a LogisticRegression")

## 4. Evaluate — Part A: Extend

*Score both models against the baseline — this closes out Part A.*

In [ ]:
from sklearn.metrics import roc_auc_score

# TODO: AUC for baseline, model, and logreg on the test set.
# YOUR CODE HERE

print(f"Baseline AUC: {baseline_auc:.3f}")
print(f"Model AUC:    {model_auc:.3f}")
print(f"Logreg AUC:   {logreg_auc:.3f}")

In [ ]:
check_in_range(model_auc, 0.75, 0.90, "model AUC is close to Lesson 8's default XGBoost result (0.83)")
check_in_range(logreg_auc, 0.75, 0.90, "logreg AUC is close to Lesson 8's logistic regression result (0.83)")
check_equal(model_auc > baseline_auc, True, "model beats the baseline")

## 5. Interpret — Part B: Decide

A retention call costs Customer Success agent time and system overhead —
call it **\$40** per outreach, a standard estimate for a short, live
retention contact. The value of a save is harder to pin down (we don't
observe how long a saved customer actually stays), so it's grounded in the
data instead of guessed: 88.5% of churners are on **month-to-month**
contracts, not locked in, so a "save" buys a bounded number of extra months
rather than a lifetime. Assume a save is worth **6 months of the churned
customer's own monthly charges**.

**Part B's answer is one number: the call threshold on XGBoost's score.**

*Find it in two passes on XGBoost's test-set predicted probabilities. First,
write a reusable `net_benefit_curve()` and sweep a coarse grid of thresholds;
expected net benefit is* `(saves x save_value) - (calls x call_cost)`.
*Second, check where the best threshold landed and refine the sweep. After
you answer, the same function runs on the logistic regression — that
comparison feeds Part C, not Part B.*

In [ ]:
call_cost = 40  # assumed: ~15 min of agent time + system/incentive overhead
save_value = 6 * df.loc[df["Churn"] == 1, "MonthlyCharges"].median()  # 6 months, at the typical churner's rate
print(f"call_cost=${call_cost}   save_value=${save_value:.0f}")

In [ ]:
thresholds = np.linspace(0.05, 0.95, 19)  # coarse first pass, every 0.05
actual_churn = (y_test == 1).to_numpy()
xgb_probs = model.predict_proba(X_test)[:, 1]

# TODO: write net_benefit_curve(probs, thresholds), returning a NumPy array of
# expected net benefit, one value per threshold: call everyone with
# probs >= threshold; each call costs call_cost, and each actual churner
# called is a save worth save_value. Then run it on xgb_probs.
# YOUR CODE HERE

coarse_best = thresholds[int(np.argmax(net_benefit))]
print(f"Coarse-grid best threshold: {coarse_best:.2f}  ->  net benefit: ${net_benefit.max():,.0f}")

In [ ]:
check_shape(net_benefit, thresholds.shape, "net_benefit_curve returns one value per threshold")
check_close(net_benefit_curve(xgb_probs, np.array([0.0]))[0],
            actual_churn.sum() * save_value - len(xgb_probs) * call_cost,
            "threshold 0 (call everyone) gives every churner saved minus every call paid for")

**Look where that landed.** If the best threshold is the lowest (or highest)
value in the grid, the sweep never saw the other side of the peak — the true
optimum could be anywhere past the edge, all the way to "call everyone."
Re-sweep a finer grid that extends past it: every 0.01 from 0.01 to 0.50.

In [ ]:
fine_thresholds = np.linspace(0.01, 0.50, 50)  # every 0.01, extending past the coarse grid's floor

# TODO: rerun net_benefit_curve() on xgb_probs over fine_thresholds, keeping
# the curve as fine_net_benefit; store the winning threshold and its net
# benefit as best_threshold / best_net_benefit.
# YOUR CODE HERE

called = xgb_probs >= best_threshold
print(f"Chosen threshold: {best_threshold:.2f}  ->  net benefit: ${best_net_benefit:,.0f}")
print(f"Calls {called.sum():,} of {len(xgb_probs):,} test customers, catching {(called & actual_churn).sum()} of {actual_churn.sum()} churners")

In [ ]:
check_shape(fine_net_benefit, fine_thresholds.shape, "fine_net_benefit covers the fine grid")
check_equal(fine_thresholds[0] < best_threshold < fine_thresholds[-1], True, "best threshold is inside the fine grid, not on its edge")
check_equal(best_net_benefit >= net_benefit.max(), True, "the finer sweep did at least as well as the coarse one")

**Your answer — call threshold on XGBoost's score:** *\<the threshold you chose\>*

**Justification (3–5 sentences):** _Your answer here._ 

**Before Part C: did the better model change the answer?** This doesn't
change your Part B answer above — it's evidence for Part C's recommendation.
Run the same fine sweep on the logistic regression's probabilities. If two
models with near-identical AUC point Customer Success at lists worth the same
dollars, the model choice is a maintenance decision, not a business one.

In [ ]:
# TODO: same fine sweep on logreg's test-set probabilities, keeping the curve
# as logreg_net_benefit; store logreg_best_threshold / logreg_best_net_benefit.
# YOUR CODE HERE

print(f"XGBoost:             threshold {best_threshold:.2f}  ->  ${best_net_benefit:,.0f}")
print(f"Logistic regression: threshold {logreg_best_threshold:.2f}  ->  ${logreg_best_net_benefit:,.0f}")

plt.plot(fine_thresholds, fine_net_benefit, label="XGBoost")
plt.plot(fine_thresholds, logreg_net_benefit, label="Logistic regression")
plt.xlabel("Call threshold (predicted churn probability)")
plt.ylabel("Expected net benefit ($)")
plt.legend()
plt.show()

In [ ]:
check_shape(logreg_net_benefit, fine_thresholds.shape, "logreg_net_benefit covers the fine grid")
check_in_range(logreg_best_threshold, fine_thresholds[0], fine_thresholds[-1], "logreg's best threshold comes from the fine grid")

## 6. Recommend — Part C: Recommend

*Stakeholder-facing. Write for the Customer Success lead who will act on
this Monday morning, not for a grader. Say which model's score the call
list should run on, using the comparison at the end of Part B.*

**Recommendation:** _Your answer here._

**What would change this recommendation:** _Your answer here._

**Limits:** _Your answer here._

## Rubric

*Same four criteria, same point split, every week — 100 pts total.*

| Criterion | Points | What it measures |
| --- | --- | --- |
| **Correctness** | 40 | Part A's checkpoints pass — the pipeline runs and produces the right shapes/values. |
| **Decision Quality** | 20 | Part B's number falls in a defensible range given the stated cost asymmetry and *your own* model's results. |
| **Justification** | 20 | Part B's reasoning (10 pts) engages with your own numbers; Part C's recommendation (10 pts) follows logically from your own Part B answer. |
| **Communication** | 20 | Part C reads as a stakeholder memo — clear, actionable, and honest about limits. |